## This notebook contains the code for the preparation of the PKU-SafeRLHF dataset ([Ji et al., 2024](https://huggingface.co/datasets/PKU-Alignment/PKU-SafeRLHF)) for experiments 1 and 2

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
from datasets import load_dataset

dataset = load_dataset("PKU-Alignment/PKU-SafeRLHF")  # full dataset version

In [ ]:
# filter for pairs with exactly one safe and one unsafe response
high_contrast = {}

for split in ["train", "test"]:
    df = dataset[split].to_pandas()
    high_contrast[split] = df[
        df["is_response_0_safe"] != df["is_response_1_safe"]
    ]

train_high_contrast = high_contrast["train"]
test_high_contrast = high_contrast["test"]

In [ ]:
# len(train_high_contrast) # 10813  # train set in original dataset: 73907
len(test_high_contrast) # 1222  # test set in original dataset: 8211
# hence roughly same train/test split of about 90/10

In [ ]:
pd.crosstab(
    train_high_contrast["response_0_severity_level"],
    train_high_contrast["response_1_severity_level"]
)

# pd.crosstab(
#     test_high_contrast["response_0_severity_level"],
#     test_high_contrast["response_1_severity_level"]
# )

In [ ]:
allowed = [0, 2, 3]  # only severity level 1 discarded

train_high_contrast_no1 = train_high_contrast[
    train_high_contrast["response_0_severity_level"].isin(allowed) &
    train_high_contrast["response_1_severity_level"].isin(allowed)
]

test_high_contrast_no1 = test_high_contrast[
    test_high_contrast["response_0_severity_level"].isin(allowed) &
    test_high_contrast["response_1_severity_level"].isin(allowed)
]

In [ ]:
# len(train_high_contrast_no1)  # 8153 (after dropna: 8141)
len(test_high_contrast_no1)  # 921
# still roughly 90/10 split

In [ ]:
train_high_contrast_no1 = train_high_contrast_no1.copy()
test_high_contrast_no1 = test_high_contrast_no1.copy()

In [ ]:
pd.crosstab(
    train_high_contrast_no1["safer_response_id"],
    train_high_contrast_no1["is_response_0_safe"]
)

# pd.crosstab(
#     test_high_contrast_no1["safer_response_id"],
#     test_high_contrast_no1["is_response_0_safe"]
# )

In [ ]:
# prepare chosen/rejected columns for DPO
train_high_contrast_no1["chosen"] = train_high_contrast_no1.apply(lambda r: r["response_0"] if r["is_response_0_safe"] else r["response_1"], axis=1)
train_high_contrast_no1["rejected"] = train_high_contrast_no1.apply(lambda r: r["response_1"] if r["is_response_0_safe"] else r["response_0"], axis=1)
test_high_contrast_no1["chosen"] = test_high_contrast_no1.apply(lambda r: r["response_0"] if r["is_response_0_safe"] else r["response_1"], axis=1)
test_high_contrast_no1["rejected"] = test_high_contrast_no1.apply(lambda r: r["response_1"] if r["is_response_0_safe"] else r["response_0"], axis=1)

In [ ]:
train_high_contrast_no1.to_csv("train_high_contrast_no1.csv", index=False)
test_high_contrast_no1.to_csv("test_high_contrast_no1.csv", index=False)